In [ ]:


import time
import tracemalloc
import random
import sys

def count_conflicts(board, n):

    conflicts = 0
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:                      # same row
                conflicts += 1
            if abs(board[i] - board[j]) == abs(i - j):   # same diagonal
                conflicts += 1
    return conflicts


def conflicts_for_queen(board, col, n):
    """
    Count how many queens attack the queen at column col.
    """
    conflicts = 0
    for c in range(n):
        if c == col:
            continue
        if board[c] == board[col]:
            conflicts += 1
        if abs(board[c] - board[col]) == abs(c - col):
            conflicts += 1
    return conflicts


def min_conflicts_row(board, col, n):

    min_conf = float('inf')
    best_rows = []

    for row in range(n):
        board[col] = row
        conf = conflicts_for_queen(board, col, n)
        if conf < min_conf:
            min_conf = conf
            best_rows = [row]
        elif conf == min_conf:
            best_rows.append(row)

    best_row = random.choice(best_rows)  # break ties randomly
    return best_row, min_conf


def hill_climbing(n, max_steps=10000):

    # Random initial placement — one queen per column, random row
    board = [random.randint(0, n - 1) for _ in range(n)]
    steps = 0

    for step in range(max_steps):
        total_conflicts = count_conflicts(board, n)

        # Solution found!
        if total_conflicts == 0:
            return board, steps, True

        # Find the column with the most conflicts (most attacked queen)
        col_conflicts = [conflicts_for_queen(board, col, n) for col in range(n)]
        max_conf = max(col_conflicts)

        # Pick a random column among those with the most conflicts
        max_cols = [c for c, conf in enumerate(col_conflicts) if conf == max_conf]
        chosen_col = random.choice(max_cols)

        # Move that queen to the row with fewest conflicts
        best_row, _ = min_conflicts_row(board, chosen_col, n)
        board[chosen_col] = best_row
        steps += 1

        # Check if we improved to zero
        if count_conflicts(board, n) == 0:
            return board, steps, True

    return None, steps, False  # stuck at local minimum


def solve_nqueens_hillclimbing(n, max_restarts=1000, max_steps=10000):

    total_steps = 0
    restarts = 0

    tracemalloc.start()
    start = time.perf_counter()

    for attempt in range(max_restarts):
        board, steps, solved = hill_climbing(n, max_steps)
        total_steps += steps
        restarts = attempt

        if solved:
            elapsed = time.perf_counter() - start
            _, peak_mem = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            memory_kb = peak_mem / 1024
            return board, elapsed, memory_kb, total_steps, restarts, True

    elapsed = time.perf_counter() - start
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    memory_kb = peak_mem / 1024
    return None, elapsed, memory_kb, total_steps, restarts, False


def validate_solution(board, n):
    """Verify no two queens attack each other."""
    if board is None:
        return False
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                return False
            if abs(board[i] - board[j]) == abs(i - j):
                return False
    return True


def print_board(board, n):
    """Print chess board for small N only."""
    border = "  +" + "---+" * n
    print(border)
    for row in range(n):
        line = "  |"
        for col in range(n):
            line += " Q |" if board[col] == row else " . |"
        print(line)
        print(border)

def run_experiments():

    test_sizes   = [10, 30, 50, 100, 200, 500]
    MAX_RESTARTS = 1000
    MAX_STEPS    = 10000

    random.seed(42)  # for reproducibility

    print("=" * 80)
    print("  N-Queens — Greedy Hill Climbing (Local Search with Restarts)")
    print("=" * 80)
    print(f"  Max restarts per run : {MAX_RESTARTS}")
    print(f"  Max steps per restart: {MAX_STEPS}")
    print("=" * 80)
    print(f"{'N':>6} | {'Time (s)':>10} | {'Memory (KB)':>12} | "
          f"{'Steps':>10} | {'Restarts':>9} | {'Valid?':>6}")
    print("-" * 80)

    results = []

    for n in test_sizes:
        print(f"  Solving N = {n} ...", end="", flush=True)

        board, elapsed, mem_kb, steps, restarts, found = solve_nqueens_hillclimbing(
            n, MAX_RESTARTS, MAX_STEPS
        )

        valid  = validate_solution(board, n)
        status = "YES" if valid else ("NO SOL" if not found else "ERROR")

        print(f"\r{n:>6} | {elapsed:>10.4f} | {mem_kb:>12.4f} | "
              f"{steps:>10,} | {restarts:>9,} | {status:>6}")

        results.append((n, elapsed, mem_kb, steps, restarts, status))

        # Show board for N=10 only
        if n == 10 and board is not None:
            print(f"\n  Solution for N=10:")
            print(f"  Queen positions (col->row): {board}")
            print_board(board, n)
            print()

    # ── SUMMARY ───────────────────────────────
    print("\n" + "=" * 80)
    print("  FINAL SUMMARY TABLE")
    print("=" * 80)
    print(f"{'N':>6} | {'Time (s)':>10} | {'Memory (KB)':>12} | "
          f"{'Steps':>10} | {'Restarts':>9} | {'Result':>6}")
    print("-" * 80)
    for row in results:
        n, t, m, st, rs, status = row
        print(f"{n:>6} | {t:>10.4f} | {m:>12.4f} | "
              f"{st:>10,} | {rs:>9,} | {status:>6}")

    print("\n  Key observations:")
    print("  - Hill Climbing solves ALL sizes including N=500.")
    print("  - Random restarts escape local minima traps.")
    print("  - Much faster than exhaustive DFS for large N.")
    print("  - Not guaranteed to find solution — may miss some cases.")
    print("=" * 80)

    return results

if __name__ == "__main__":
    run_experiments()

  N-Queens — Greedy Hill Climbing (Local Search with Restarts)
  Max restarts per run : 1000
  Max steps per restart: 10000
     N |   Time (s) |  Memory (KB) |      Steps |  Restarts | Valid?
--------------------------------------------------------------------------------
    10 |     0.0026 |       0.6094 |         12 |         0 |    YES

  Solution for N=10:
  Queen positions (col->row): [3, 0, 4, 7, 9, 2, 6, 8, 1, 5]
  +---+---+---+---+---+---+---+---+---+---+
  | . | Q | . | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+
  | . | . | . | . | . | . | . | . | Q | . |
  +---+---+---+---+---+---+---+---+---+---+
  | . | . | . | . | . | Q | . | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+
  | Q | . | . | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+
  | . | . | Q | . | . | . | . | . | . | . |
  +---+---+---+---+---+---+---+---+---+---+
  | . | . | . | . | . | . | . | . | . | Q |
  +---+---+---+---+---+---+---+---+---+---+
  

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

# Your experimental data points
N = [10, 30, 50, 100, 200, 500]
time_taken = [0.0026, 0.1450, 1.2583, 6.6071, 27.6470, 595.3838]

# Configure the plot
plt.figure(figsize=(7, 4.5))
plt.plot(N, time_taken, marker='o', color='#1f77b4', linestyle='-', linewidth=2, label='Execution Time')

# Style adjustments
plt.xlabel('Board Size (N)', fontsize=11)
plt.ylabel('Time (seconds)', fontsize=11)
plt.title('Greedy Hill Climbing Analysis: Board Size vs. Time', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()

# 1. Save the graph directly as a PDF file
pdf_filename = 'time_complexity_plot.pdf'
plt.savefig(pdf_filename, format='pdf', dpi=300)
plt.show()

# 2. Trigger the automatic download helper in your browser
files.download(pdf_filename)
